In [ ]:
%load_ext autoreload
%autoreload 2

from datetime import datetime, timedelta, date
from functools import partial
import numpy as np
import polars as pl

from okx.store import OrderbookStore
from okx.recipes.forwards import build_forwards_pchip, build_forwards_kalman, prepare_pillars
from forwards.pchip import reconstruct_forward, PCHIPCurve
from forwards.kalman_ns import reconstruct_ns_forward, NSCarryState
from forwards.evaluation import wmae_pillar_fit, leave_one_expiry_out, evaluate_curve_snapshot

In [ ]:
store = OrderbookStore(
    data_root="data/okx",
    manifest_path="data/okx/manifest.sqlite",
    batch_days=5
)


In [ ]:
store.clear_cache()

In [ ]:
# Fix: Convert datetime objects to date objects
date = [date(2025, 9, 2)]

swap_lf = store.get(
    inst_type="SWAP",
    inst_family="BTC-USD",
    dates=date,
    depth=1,
    features=['trim', 'strip']
)
futures_lf = store.get(
    inst_type="FUTURES",
    inst_family="BTC-USD",
    dates=date,
    depth=1,
    features=['trim', 'strip']
)
options_lf = store.get(
    inst_type="OPTION",
    inst_family="BTC-USD",
    dates=date,
    depth=1,
    features=['trim', 'strip']
)

print("Done")

swap_df=swap_lf.collect()
futures_df=futures_lf.collect()
options_df=options_lf.collect()

from datetime import datetime

def show_df_time_range(name, df):
    if df.is_empty():
        print(f"{name}: DataFrame is empty")
        return
    timeMs = df['timeMs'].to_numpy()
    earliest = timeMs.min()
    latest = timeMs.max()
    if hasattr(earliest, 'item'):
        earliest = earliest.item()
    if hasattr(latest, 'item'):
        latest = latest.item()
    earliest_dt = datetime.utcfromtimestamp(earliest / 1000)
    latest_dt = datetime.utcfromtimestamp(latest / 1000)
    print(f"{name}:")
    print(f"  Earliest timeMs: {earliest} ({earliest_dt})")
    print(f"  Latest   timeMs: {latest} ({latest_dt})\n")

show_df_time_range("swap_df", swap_df)
show_df_time_range("futures_df", futures_df)
show_df_time_range("options_df", options_df)
